# 시작하기 · PhysicsNeMo 실습

**Brev의 준비된 Jupyter 환경에서 실행합니다.** Gemini 또는 ChatGPT는 평소처럼 브라우저에서 엽니다. 이 둘을 서로 연결하거나 API key를 입력할 필요가 없습니다.

1. 이 노트북에서 환경을 확인합니다.
2. [PINN 실습](01_PINN.ipynb)에서 학습 결과를 그리고 봅니다.
3. [FNO 실습](02_FNO.ipynb)에서 두 함수를 완성하고 실행합니다.

코드 셀은 선택한 뒤 `Shift + Enter`로 실행합니다. 실행 중인 셀은 `[*]`로 표시됩니다.


## 1. 교재 폴더와 커널 확인


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pinn_basics.py").is_file()
             and (p / "02_FNO.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("워크숍 교재 폴더 안에서 이 노트북을 여세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("교재:", ROOT)
print("Python:", sys.executable)


In [ ]:
import importlib.metadata as meta
import importlib.util
import platform

expected = {
    "nvidia-physicsnemo": "2.2.2", "torch": "2.10.0+cu128",
    "numpy": "2.5.2", "sympy": "1.14.0", "PyYAML": "6.0.3",
    "h5py": "3.16.0", "matplotlib": "3.11.2",
}
print("Python:", platform.python_version(), "(원본 기준: 3.12)")
missing = []
for package, target in expected.items():
    try:
        actual = meta.version(package)
        print(f"{package}: {actual}" + (" ✓" if actual == target else f" — 원본 기준 {target}; 강사 확인"))
    except meta.PackageNotFoundError:
        missing.append(package)
if missing:
    raise RuntimeError("준비된 course 커널을 선택하세요. 없는 패키지: " + ", ".join(missing))

import torch
from physicsnemo.models.mlp import FullyConnected
from physicsnemo.models.fno import FNO
from physicsnemo.sym.eq.phy_informer import PhysicsInformer
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU를 사용할 수 없습니다. 커널과 Brev 환경을 강사와 확인하세요.")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)
x = torch.tensor([1.0], device="cuda", requires_grad=True)
(x*x).sum().backward()
assert torch.isfinite(x.grad).all()
print("GPU tensor와 미분 확인 완료")


버전이 다르거나 오류가 있으면 강사에게 화면을 보여 주세요. 여기서 패키지를 새로 설치하지 않습니다. 위 확인은 커널과 GPU의 기본 동작 확인이며 실습의 학습 완료를 뜻하지 않습니다.

## 2. FNO 데이터 준비

보통 강사가 미리 준비합니다. 다음 셀은 기존 데이터가 있으면 검사하고, 없으면 원본의 기본 데이터셋을 한 번 생성합니다. 처음 생성할 때는 잠시 걸릴 수 있습니다. 준비가 끝나면 두 실습에서 같은 데이터와 설정을 사용합니다.


In [ ]:
import os
import subprocess
from src.operator_training import load_data
DATA_DIR = ROOT / "data/reaction_diffusion"
files = [DATA_DIR / f"{name}.hdf5" for name in ("train", "val", "test")]
if any(p.exists() for p in files) and not all(p.exists() for p in files):
    raise RuntimeError("일부 데이터 파일만 있습니다. 기존 파일을 보존하고 강사에게 확인하세요.")
if not all(p.exists() for p in files):
    env = dict(os.environ, OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
    subprocess.run([sys.executable, "-m", "src.generate_data", "--seed", "42",
                    "--output-dir", str(DATA_DIR)], cwd=ROOT, env=env, check=True)
pairs = load_data(DATA_DIR)
print({name: tuple(pair[0].shape) for name, pair in zip(("train", "validation", "test"), pairs)})
del pairs
print("데이터 준비 완료:", DATA_DIR)


## 준비 완료

강사 설명에 맞춰 [01_PINN.ipynb](01_PINN.ipynb)를 여세요. 데이터가 생성되는 동안에는 PINN 이론 설명을 먼저 들을 수 있습니다.

| 시간 | 진행 |
| --- | --- |
| 0–10분 | 환경 준비 |
| 10–35분 | PINN 원리 |
| 35–55분 | PINN 실습 |
| 55–75분 | FNO 원리 |
| 75–105분 | FNO 실습 |
| 105–120분 | 결과 비교와 질문 |
